# Lesson 2.3: value_counts aur groupby

Cricket mein dekha jata hai: *"Kis player ka average sab se zyada?"*, *"Kis team ke khilaf batting mushkil?"*. Ye exactly **groupby** hai.

| Tool | Kaam | SQL mein |
|---|---|---|
| `value_counts()` | ek categorical column mein **har category kitni dafa** | `SELECT col, COUNT(*) ... GROUP BY col` |
| `groupby()` | data ko **groups** mein baanto, har group ki statistics | `SELECT player, AVG(runs) ... GROUP BY player` |

**ML ke liye kyun:**
- `value_counts()` → encoding se pehle check: kitni categories, koi **bohat kam** to nahi aati?
- `groupby()` → naye **features** bante hain (player ka career average, recent form)

In [ ]:
import pandas as pd

data = {
    "player":           ["Babar", "Rizwan", "Babar", "Fakhar", "Rizwan", "Babar", "Fakhar", "Rizwan"],
    "opposition":       ["India", "Australia", "England", "India", "England", "Australia", "England", "India"],
    "batting_position": [3, 1, 3, 2, 1, 3, 2, 1],
    "balls":            [30, 50, 15, 60, 28, 3, 70, 40],
    "fours":            [5, 8, 1, 10, 3, 0, 12, 4],
    "sixes":            [1, 2, 0, 4, 1, 0, 3, 2],
    "runs":             [45, 67, 12, 89, 34, 0, 102, 51],
}

df = pd.DataFrame(data)
df

---
## Part 1: value_counts, nunique, unique

Har category aur uski ginti, **zyada se kam** order mein.

In [ ]:
df["player"].value_counts()

In [ ]:
df["opposition"].value_counts()

| Command | Kya deta hai |
|---|---|
| `nunique()` | **kitni** alag categories (ek number) |
| `unique()` | **kaun si** categories (list) |

Lesson 2.4 mein `unique()` ne hi `india` aur `India ` jaise chhupe masle pakre the.

In [ ]:
print(df["player"].nunique())
print(df["player"].unique())

---
## Part 2: groupby, ek statistic

```
df.groupby("player")   →  rows ko player ke hisaab se groups mein baanto
["runs"]               →  har group mein sirf runs column lo
.mean()                →  har group ka average
```

Andar kya hota hai:
```
Babar:   45, 12, 0     →  mean 19.0
Fakhar:  89, 102       →  mean 95.5
Rizwan:  67, 34, 51    →  mean 50.67
```

In [ ]:
df.groupby("player")["runs"].mean()

Result mein left side pe **player ke naam** hain, 0–7 numbers nahi. **groupby ke baad group ka naam index ban jata hai.** (Charts mein `.index` aur `.values` isi liye kaam aate hain.)

In [ ]:
df.groupby("batting_position")["runs"].mean()

⚠️ Ye **batting_position** wala result player wale result jaisa hi hai (19, 95.5, 50.67)! Kyun? Har position pe **sirf ek player** hai (1 = Rizwan, 2 = Fakhar, 3 = Babar). To ye "position ka asar" nahi, balke **player ka asar** hai. (Yahi baat correlation −0.38 mein bhi aayi thi.)

---
## Part 3: agg, kai statistics ek saath

`.agg([...])` → har group ki ek row, har statistic ka ek column. Result ek chhota **DataFrame**.

In [ ]:
df.groupby("player")["runs"].agg(["count", "sum", "mean", "max"])

**`count` hamesha saath rakhein.** Fakhar ka average 95.5 sirf **2 innings** se bana hai. Itne kam data pe bharosa nahi kar sakte.

---
## Part 4: do columns pe group
Har player, **har opposition** ke khilaf. Result ka index ab do levels ka hai (player → opposition).

In [ ]:
df.groupby(["player", "opposition"])["runs"].sum()

---
## Part 5: Player ka asal (overall) strike rate

Lesson 1.1 yaad karein: **strike rates ka average** aur **overall strike rate** alag hote hain. Cricket mein **overall** sahi hai:
```
total runs ÷ total balls × 100
```
Isliye pehle har player ke **total** runs aur balls jodo, phir divide karo.

In [ ]:
totals = df.groupby("player")[["runs", "balls"]].sum()
totals["strike_rate"] = (totals["runs"] / totals["balls"] * 100).round(2)
totals.sort_values("strike_rate", ascending=False)

Index wapas normal column banana ho (masalan CSV save karne ya merge ke liye):

In [ ]:
totals.reset_index()

---
## Part 6: Exercise ke jawab (har sawal alag cell mein)

⚠️ Aapke purane notebook mein saare 5 sawal **ek cell** mein the, isliye sirf **aakhri** ka output dikha. Notebook sirf cell ki **aakhri line** khud dikhati hai.

### Sawal 1: Har opposition ke khilaf kitni innings?
Do tareeqe, same jawab:

In [ ]:
df["opposition"].value_counts()

In [ ]:
df.groupby("opposition").size()

### Sawal 2: Har player ke total runs, zyada se kam

In [ ]:
df.groupby("player")["runs"].sum().sort_values(ascending=False)

### Sawal 3: Har opposition ke khilaf average runs

In [ ]:
df.groupby("opposition")["runs"].agg(["mean", "count"]).round(2).sort_values("mean")

**Jawab:** data ke hisaab se **Australia** sab se mushkil (33.5). Lekin dekhein: ye **sirf 2 innings** se bana hai, aur unmein ek **duck** hai.

> **Small sample size:** groupby ka result dekhein to **count** hamesha saath dekhein. Client aksar aise numbers se bare faisle karna chahte hain; aapka kaam khatra batana hai.

### Sawal 4: Har player ki kitni innings 50+?
`fifty_plus` True/False column hai, aur Python mein **True = 1, False = 0**. To groupby ke baad `.sum()` = **ginti**.

In [ ]:
df["fifty_plus"] = df["runs"] >= 50
df.groupby("player")["fifty_plus"].sum()

### Sawal 5: Har player ka count, mean, median, max

In [ ]:
df.groupby("player")["runs"].agg(["count", "mean", "median", "max"]).round(2)

Babar: mean 19 lekin median **12**. 45 wali innings mean ko upar kheench rahi hai.

---
## Part 7: Leakage — groupby se feature banate waqt

**Galat feature:** `player_avg` = har player ka **in 8 innings** ka average, aur usay X mein daal ke **inhi** innings ke runs predict karna.

Babar ki pehli innings (45) predict karte waqt `player_avg` = (45 + 12 + 0) / 3 = 19. Do masle:
1. **45 khud** feature mein hai → jawab andar chhupa hai
2. **12 aur 0 future** ki innings hain → match se pehle maloom nahi ho sakti

Isay **target leakage** kehte hain: training mein achha, naye match pe fail.

In [ ]:
# ❌ Ghalat: saari innings ka average, har row pe
df["player_avg_wrong"] = df.groupby("player")["runs"].transform("mean")
df[["player", "runs", "player_avg_wrong"]]

`transform("mean")` = groupby ka result **har row pe wapas** likh do (agg ki tarah chhoti table nahi banata).

### ✅ Sahi: sirf pichli innings ka average
- **`shift(1)`**: har innings ko ek qadam neeche khiskao, taake har row ko sirf **pichli** innings dikhein, khud apni nahi
- **`expanding().mean()`**: ab tak ki **saari pichli** innings ka average

(Ye code sirf dekhne ke liye hai, yaad karne ki zaroorat nahi.)

In [ ]:
df["player_prev_avg"] = (
    df.groupby("player")["runs"]
      .transform(lambda s: s.shift(1).expanding().mean())
)
df[["player", "runs", "player_avg_wrong", "player_prev_avg"]]

### Babar ke liye

| Innings | Runs | Galat avg | Sahi (pichli) avg | Kyun |
|---|---|---|---|---|
| Pehli | 45 | 19 | **NaN** | koi pichli innings nahi |
| Doosri | 12 | 19 | **45.0** | sirf pehli |
| Teesri | 0 | 19 | **28.5** | (45 + 12) / 2 |

Do baatein:
- **Pehli innings mein NaN**: ye missing value **hamare feature engineering** ne banayi ("debut" ke liye kya rakhein? faisla karna hoga)
- Ye tab hi sahi hai jab rows **date ke order** mein hon. Real data mein pehle `sort_values("date")` lazmi.

---
## Khulasa
| Kaam | Code |
|---|---|
| Har category kitni dafa | `value_counts()` |
| Kitni / kaun si categories | `nunique()` / `unique()` |
| Har group ka ek number | `groupby("col")["x"].mean()` |
| Kai numbers | `.agg(["count", "mean", ...])` |
| Do levels | `groupby(["a", "b"])` |
| Har row pe wapas | `.transform(...)` |
| Index → column | `.reset_index()` |

**Yaad rakhein:** count saath dekho · True/False ka `.sum()` = ginti · groupby features mein **future data** na ghuse